## 0. Preparación

In [1]:
import json
import random
import re
import time
from datetime import date, datetime, timezone
from enum import Enum
from pathlib import Path
from typing import Any

import pandas as pd
from pydantic import BaseModel, Field
from pydantic_ai import Agent

from renewables_permitting.utils import (
    normalize_text,
    save_parquet,
    validate_required_columns,
)

BASE_URL = "https://www.boe.es/datosabiertos/api/boe/sumario"

# PROJECT_ROOT = Path(__file__).resolve().parents[2]  # fuera del notebook
PROJECT_ROOT = Path.cwd().parent  # dentro del notebook

DATA_DIR = PROJECT_ROOT / "data"

BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"

# BRONZE
BOE_DOCS_XML_DIR = BRONZE_DIR / "boe_docs_xml"


# SILVER
BOE_CANDIDATES_PATH = SILVER_DIR / "boe_candidates" / "boe_candidates_normalized.parquet"

BOE_CANDIDATES_DOCS_TEXT_PATH = SILVER_DIR / "boe_candidates_docs_text" / "boe_candidates_docs_text.parquet"

DIM_MUNICIPALITIES_PATH = SILVER_DIR / "dimensions" / "dim_municipalities.parquet"

SILVER_BOE_AI_DIR = SILVER_DIR / "boe_ai"

BOE_AI_EXTRACTIONS_PATH = SILVER_BOE_AI_DIR / "boe_ai_extractions.parquet"
LIFECYCLE_EVENTS_PATH = SILVER_BOE_AI_DIR / "lifecycle_events.parquet"
ADMINISTRATIVE_ACTIONS_PATH = SILVER_BOE_AI_DIR / "administrative_actions.parquet"
ASSET_MENTIONS_PATH = SILVER_BOE_AI_DIR / "asset_mentions.parquet"
ASSET_TECHNOLOGIES_PATH = SILVER_BOE_AI_DIR / "asset_technologies.parquet"
ASSET_PARTICIPANTS_PATH = SILVER_BOE_AI_DIR / "asset_participants.parquet"
ASSET_LOCATIONS_PATH = SILVER_BOE_AI_DIR / "asset_locations.parquet"
ASSET_ALIASES_PATH = SILVER_BOE_AI_DIR / "asset_aliases.parquet"
ASSET_RELATION_MENTIONS_PATH = SILVER_BOE_AI_DIR / "asset_relation_mentions.parquet"


# GOLD
PROJECT_GROUPS_PATH = GOLD_DIR / "project_groups.parquet"
PROJECT_ASSETS_PATH = GOLD_DIR / "project_assets.parquet"
PROJECT_TIMELINE_PATH = GOLD_DIR / "project_timeline.parquet"
PROJECT_STATUS_PATH = GOLD_DIR / "project_status.parquet"

## 1. Instrucciones

In [ ]:
INSTRUCTIONS = """
Eres un extractor canónico de información estructurada de documentos del BOE sobre proyectos energéticos.
Devuelve exclusivamente JSON válido conforme al esquema BOEProjectExtraction.

Objetivo:
Extraer eventos de ciclo de vida de proyectos energéticos a partir de una publicación del BOE.

Reglas generales:
- Extrae únicamente información explícitamente contenida en el título o en el texto del documento.
- No inventes, completes, deduzcas ni corrijas información por conocimiento externo.
- Si un dato no aparece, usa null o lista vacía según corresponda al esquema.
- No utilices valores literales como "no_consta", "desconocido" o equivalentes salvo que formen parte explícita del documento.
- Toda información extraída debe estar respaldada por evidencia textual.
- Si existe ambigüedad, conserva la información observada y no fuerces una interpretación.
- Limita la extracción a la publicación actual.
- No intentes determinar si varias publicaciones pertenecen al mismo proyecto global.
- No asignes identificadores definitivos de proyecto.
- Usa únicamente identificadores locales internos al documento: asset_1, asset_2, asset_3, etc.
- Distingue entre hechos publicados y antecedentes históricos mencionados como contexto.
- event_summary debe contener una descripción breve y no nula del hecho principal publicado.

Interpretación numérica:
- Interpreta los números con formato español.
- "31,172 MW" equivale a 31.172 MW.
- "28.000 kW" equivale a 28000 kW.
- Cuando el texto incluya potencia unitaria y número de equipos, calcula la potencia total normalizada si la equivalencia es explícita.
- Si existe discrepancia aparente entre el cálculo explícito y una cifra textual, no afirmes que el BOE contiene una errata.
- Conserva la discrepancia en power_normalization_note.
- Ejemplo: "14 aerogeneradores de 2000 kW" se normaliza como 28 MW; si el texto además dice "28.000 MW", indícalo en la nota sin corregir ni calificar el texto.

Relevancia:
- Clasifica como relevante solo documentos vinculados a proyectos energéticos concretos.
- Un documento es relevante si trata sobre generación eléctrica renovable, almacenamiento, infraestructuras de evacuación, líneas eléctricas, subestaciones, autorizaciones administrativas, evaluación ambiental, declaración de impacto ambiental, informe de determinación de afección ambiental, declaración de utilidad pública, expropiación, archivo, denegación, modificación, ampliación o repotenciación de un proyecto concreto.
- Clasifica como no_relevante documentos normativos, estadísticos, tarifarios, presupuestarios, genéricos o no vinculados a un proyecto energético identificable.
- Usa dudoso cuando exista vocabulario energético pero no haya información suficiente para identificar un proyecto tramitado.

Unidad de extracción:
- La unidad principal de extracción es lifecycle_events.
- Cada publicación BOE debe generar normalmente un único lifecycle_event principal.
- Un lifecycle_event representa una evolución material o funcional de un activo energético y los actos administrativos publicados sobre ella.
- Solo crea varios lifecycle_events cuando la publicación describa evoluciones materiales independientes o afecte a activos principales distintos.
- No crees un lifecycle_event separado para un trámite administrativo que ya esté representado en administrative_actions.
- Si una publicación contiene varios actos administrativos sobre la misma evolución del proyecto o activo, inclúyelos como administrative_actions dentro del mismo lifecycle_event.
- No generes lifecycle_events adicionales para representar antecedentes históricos, contexto administrativo o referencias a procedimientos previos.
- No añadas varios lifecycle_events para describir el mismo hecho administrativo.
- No utilices event_type = other o unknown cuando el hecho principal pueda clasificarse razonablemente mediante alguno de los tipos definidos en el esquema.

Procedimiento administrativo:
- administrative_actions debe recoger los actos administrativos publicados en el documento.
- Cada administrative_action debe incluir procedure_stage, procedure_decision y evidence.
- procedure_stage debe reflejar el trámite administrativo publicado en el BOE.
- procedure_decision debe reflejar la decisión principal asociada al trámite: formulado, favorable, desfavorable, autorizado, denegado, sometido_informacion_publica, declarado_utilidad_publica, archivado, desistido, inadmitido u otra decisión explícitamente publicada.
- Si una misma publicación contiene varios actos administrativos sobre la misma evolución del proyecto o activo, inclúyelos como administrative_actions dentro del mismo lifecycle_event.
- No dupliques como lifecycle_event lo que ya esté representado mediante procedure_stage o procedure_decision.
- No conviertas antecedentes históricos en administrative_actions del evento actual salvo que el documento los publique nuevamente como parte del acto administrativo objeto de la resolución.
- Los antecedentes pueden utilizarse para contextualizar el evento, pero no deben generar nuevos lifecycle_events ni nuevas administrative_actions.
- Cuando existan varios trámites o decisiones publicados en el mismo documento, extrae todos los que formen parte del acto administrativo objeto de publicación.
- Cada administrative_action debe incluir evidence específica que justifique el trámite y la decisión extraídos.

Activos:
- Extrae únicamente los activos necesarios para entender el hecho administrativo publicado.
- El asset principal debe corresponder al activo directamente afectado por el procedimiento.
- Un mismo lifecycle_event puede contener varios assets cuando todos formen parte de la misma actuación administrativa publicada.
- No crees assets duplicados para representar distintas fases administrativas del mismo activo.
- Extrae nuevas instalaciones de generación.
- Extrae instalaciones existentes afectadas por modificaciones, ampliaciones, repotenciaciones o hibridaciones.
- Extrae sistemas de almacenamiento.
- Extrae infraestructuras energéticas únicamente cuando constituyan el objeto principal del procedimiento.
- No extraigas infraestructuras auxiliares de evacuación, conexión o acceso a red como assets independientes salvo que constituyan el objeto principal del documento.
- No extraigas como assets independientes proyectos mencionados únicamente como contexto, antecedentes, agrupaciones, complejos energéticos, comparaciones o referencias informativas.
- Si otros proyectos se mencionan únicamente para indicar pertenencia a un mismo conjunto o complejo, no los extraigas como assets salvo que exista una actuación administrativa, técnica o territorial específica sobre ellos.
- Cuando un activo aparezca únicamente en una enumeración, listado, agrupación o complejo energético y no exista una actuación administrativa, técnica o territorial específica sobre él, no lo extraigas como asset.
- Cuando exista duda, prioriza siempre el activo objeto del procedimiento frente a activos mencionados de forma contextual.
- Asocia la potencia a un asset únicamente cuando el documento la atribuya explícitamente a dicho activo.
- No transfieras potencias entre activos relacionados.
- No deduzcas la potencia de un asset a partir de la potencia de un complejo, agrupación o conjunto de proyectos salvo que el documento lo indique expresamente.
- Cada asset debe incluir evidence específica del propio activo.
- La evidence debe justificar la existencia, identificación, potencia o papel del activo en el evento.
- No utilices una cita genérica de toda la resolución como evidence de varios assets distintos.

Relaciones entre activos:
- Crea asset_relations únicamente cuando ambos activos desempeñen un papel material en el evento descrito.
- No crees relaciones para describir infraestructuras auxiliares de evacuación, conexión o acceso.
- No crees relaciones cuando uno de los activos aparezca únicamente como contexto.
- Usa:
  - hybridizes_with
  - adds_technology_to
  - adds_storage_to
  - modifies
  - expands
  - repowers
  - same_project_group_as
  - associated_with
- Usa same_project_group_as únicamente cuando el documento indique explícitamente que varios activos forman parte de un mismo proyecto, complejo o agrupación energética.

Hibridación:
- Extrae la nueva instalación incorporada mediante la hibridación.
- Extrae también el activo existente afectado cuando aparezca explícitamente en el documento.
- Crea una relación hybridizes_with entre ambos activos.
- Añade adds_technology_to cuando el documento indique expresamente la incorporación de una nueva tecnología a un activo o complejo existente.
- No extraigas infraestructuras auxiliares de conexión o evacuación como activos de la hibridación.

Promotores y participantes:
- Extrae promotores, copromotores, titulares u operadores solo si aparecen explícitamente.
- Puede haber varios participantes.
- No asumas que el promotor de un activo es también promotor de otro si el texto no lo dice.
- Conserva la denominación literal de la entidad.
- No inventes CIF, NIF ni identificadores societarios.
- No resuelvas entidades empresariales por conocimiento externo.

Localizaciones:
- Extrae únicamente municipios, provincias y comunidades autónomas explícitamente mencionados.
- Conserva la forma textual utilizada en el documento.
- No normalices nombres ni generes códigos administrativos.
- Usa province_hint y autonomous_community_hint únicamente cuando el documento proporcione dicha información.
- Si el documento identifica municipios afectados, términos municipales afectados o expresiones equivalentes asociadas al procedimiento principal, asocia dichas localizaciones al asset principal.
- No asocies localizaciones a activos mencionados únicamente como contexto.
- Incluye evidencia textual siempre que sea posible.
"""

## 2. Contratos de salida

Qué quiero saber de cada publicación?

### Clasificación documental

In [3]:
# Clasificación global de relevancia energética.

class RelevanciaEnergetica(str, Enum):
    RELEVANTE = "relevante"
    NO_RELEVANTE = "no_relevante"
    DUDOSO = "dudoso"
    
# relevante:
#   Documento sobre generación eléctrica renovable, almacenamiento, evacuación, subestaciones, líneas eléctricas o autorizaciones ambientales/administrativas asociadas.

# no_relevante:
#   Documento energético genérico, normativo, estadístico, tarifario, presupuestario o no vinculado a un proyecto concreto.

# dudoso:
#   Documento con vocabulario energético, pero sin información suficiente para saber si corresponde a un proyecto tramitado.

### Tecnologías e instalaciones energéticas

In [4]:
# Tecnologías de generación y almacenamiento.
class TechnologyType(str, Enum):
    FOTOVOLTAICA = "fotovoltaica"
    EOLICA = "eolica"
    TERMOSOLAR = "termosolar"
    HIDROELECTRICA = "hidroelectrica"
    GEOTERMICA = "geotermica"
    BIOMASA = "biomasa"
    BIOGAS = "biogas"
    HIDROGENO_VERDE = "hidrogeno_verde"
    ALMACENAMIENTO = "almacenamiento"
    OTRA = "otra"
    DESCONOCIDA = "desconocida"

In [5]:
# Características técnicas de una tecnología.
class Technology(BaseModel):
    technology_type: TechnologyType
    installed_power_mw: float | None = None
    peak_power_mwp: float | None = None
    description: str | None = None
    power_normalization_note: str | None = None

In [6]:
# Sistemas de almacenamiento energético.
class StorageSystem(BaseModel):
    exists: bool = False
    power_mw: float | None = None
    capacity_mwh: float | None = None
    description: str | None = None

In [7]:
# class InfrastructureType(str, Enum):
#     SUBESTACION = "subestacion"
#     LINEA_ELECTRICA = "linea_electrica"
#     CENTRO_SECCIONAMIENTO = "centro_seccionamiento"
#     INFRAESTRUCTURA_EVACUACION = "infraestructura_evacuacion"
#     OTRA = "otra"
#     DESCONOCIDA = "desconocida"

# class Infrastructure(BaseModel):
#     type: InfrastructureType
#     name: str | None = None
#     voltage_kv: float | None = None
#     length_km: float | None = None

In [8]:
# class HybridConfiguration(BaseModel):
#     is_hybrid: bool = False
#     generation_technology_types: list[TechnologyType] = Field(default_factory=list)
#     includes_storage: bool = False
#     description: str | None = None

### Localización administrativa (INE)

Usar Pydantic AI solo para extraer candidatos textuales y contexto; la validación final debe hacerla una función determinista.

In [9]:
class ExtractedMunicipalityMention(BaseModel):
    municipality_name: str
    province_hint: str | None = None
    autonomous_community_hint: str | None = None
    evidence: str | None = None

In [10]:
# TODO: Voy a dejar fuera de la IA la asignación determinista de los códigos INE

# # Estado de resolución administrativa.
# class MunicipalityResolutionStatus(str, Enum):
#     RESOLVED = "resolved"
#     AMBIGUOUS = "ambiguous"
#     NOT_FOUND = "not_found"


# # Municipio normalizado mediante catálogo INE.
# class MunicipalityLocation(BaseModel):
#     ine_municipality_code: str
#     municipality: str
#     ine_province_code: str | None = None
#     province: str | None = None
#     ine_autonomous_community_code: str | None = None
#     autonomous_community: str | None = None
    

# # Resultado de resolución de municipios.    
# class MunicipalityLookupResult(BaseModel):
#     query: str
#     province_hint: str | None = None
#     autonomous_community_hint: str | None = None

#     resolution_status: MunicipalityResolutionStatus
#     resolved: MunicipalityLocation | None = None
#     candidates: list[MunicipalityLocation] = Field(default_factory=list)

#     matched_by: str | None = None
#     reason: str | None = None

### Procedimiento administrativo

In [11]:
# Tipo de trámite administrativo.
class ProcedureStage(str, Enum):
    # Inicio y antecedentes del procedimiento
    SOLICITUD_TRAMITACION = "solicitud_tramitacion"
    SOLICITUD_TRAMITACION_AMBIENTAL = "solicitud_tramitacion_ambiental"
    SUBSANACION_DOCUMENTACION = "subsanacion_documentacion"
    VERIFICACION_REQUISITOS_TRAMITACION = "verificacion_requisitos_tramitacion"

    # Información pública
    INFORMACION_PUBLICA = "informacion_publica"

    # Evaluación ambiental
    DECLARACION_IMPACTO_AMBIENTAL = "declaracion_impacto_ambiental"
    INFORME_DETERMINACION_AFECCION_AMBIENTAL = "informe_determinacion_afeccion_ambiental"

    # Autorizaciones energéticas
    AUTORIZACION_ADMINISTRATIVA_PREVIA = "autorizacion_administrativa_previa"
    AUTORIZACION_ADMINISTRATIVA_CONSTRUCCION = "autorizacion_administrativa_construccion"
    AUTORIZACION_EXPLOTACION = "autorizacion_explotacion"

    # Utilidad pública y expropiación
    DECLARACION_UTILIDAD_PUBLICA = "declaracion_utilidad_publica"
    EXPROPIACION_FORZOSA = "expropiacion_forzosa"
    RELACION_BIENES_DERECHOS_AFECTADOS = "relacion_bienes_derechos_afectados"
    LEVANTAMIENTO_ACTAS_PREVIAS_OCUPACION = "levantamiento_actas_previas_ocupacion"
    ACTAS_OCUPACION = "actas_ocupacion"

    # Modificaciones y terminación anormal
    MODIFICACION = "modificacion"
    ARCHIVO_EXPEDIENTE = "archivo_expediente"
    DESISTIMIENTO = "desistimiento"
    INADMISION = "inadmision"

    # Fallback
    OTRO = "otro"
    NO_CONSTA = "no_consta"

In [12]:
# Resultado o decisión administrativa.
class ProcedureDecision(str, Enum):
    # Inicio y tramitación del expediente
    SOLICITADO = "solicitado"
    SUBSANADO = "subsanado"
    REQUISITOS_VERIFICADOS = "requisitos_verificados"
    MODIFICADO = "modificado"
    PRORROGADO = "prorrogado"
    FORMULADO = "formulado"

    # Información pública y participación
    SOMETIDO_INFORMACION_PUBLICA = "sometido_informacion_publica"
    CONVOCADO = "convocado"

    # Evaluación ambiental
    FAVORABLE = "favorable"
    DESFAVORABLE = "desfavorable"
    SOMETIDO_EIA_ORDINARIA = "sometido_eia_ordinaria"
    NO_SOMETIDO_EIA_ORDINARIA = "no_sometido_eia_ordinaria"
    DECLARADO_UTILIDAD_PUBLICA = "declarado_utilidad_publica"

    # Resolución administrativa
    APROBADO = "aprobado"
    AUTORIZADO = "autorizado"
    DENEGADO = "denegado"

    # Terminación anormal del procedimiento
    ARCHIVADO = "archivado"
    DESISTIDO = "desistido"
    INADMITIDO = "inadmitido"

    # Información no disponible
    NO_CONSTA = "no_consta"

In [13]:
# Acto administrativo publicado en el BOE.
class AdministrativeAction(BaseModel):
    procedure_stage: ProcedureStage = ProcedureStage.NO_CONSTA  # "el campo tiene tipo ProcedureStage y valor por defecto NO_CONSTA"
    procedure_decision: ProcedureDecision = ProcedureDecision.NO_CONSTA
    evidence: str | None = None

### Participantes

In [14]:
# Rol de una entidad participante.
class ParticipantRole(str, Enum):
    PROMOTER = "promoter"
    CO_PROMOTER = "co_promoter"
    OWNER = "owner"
    OPERATOR = "operator"
    GRID_OWNER = "grid_owner"
    ADMINISTRATION = "administration"
    UNKNOWN = "unknown"

In [15]:
# Empresa, administración o entidad participante.
class ProjectParticipant(BaseModel):
    name: str
    role: ParticipantRole = ParticipantRole.UNKNOWN
    evidence: str | None = None

### Activos energéticos

In [16]:
# Papel del activo dentro del evento.
class AssetRole(str, Enum):
    NEW_ASSET = "new_asset"
    EXISTING_ASSET = "existing_asset"
    MODIFIED_ASSET = "modified_asset"
    AFFECTED_ASSET = "affected_asset"
    ASSOCIATED_ASSET = "associated_asset"
    MAIN_ASSET = "main_asset"
    UNKNOWN = "unknown"

In [17]:
# Estado administrativo u operativo del activo.
class AssetStatus(str, Enum):
    PLANNED = "planned"
    UNDER_PERMITTING = "under_permitting"
    AUTHORIZED = "authorized"
    UNDER_CONSTRUCTION = "under_construction"
    IN_OPERATION = "in_operation"
    EXISTING = "existing"
    DENIED = "denied"
    ARCHIVED = "archived"
    UNKNOWN = "unknown"

In [18]:
# Instalación energética identificada en el documento.
class EnergyAsset(BaseModel):
    local_asset_id: str
    name: str | None = None
    aliases: list[str] = Field(default_factory=list)

    role_in_event: AssetRole = AssetRole.UNKNOWN
    status_in_document: AssetStatus = AssetStatus.UNKNOWN

    technologies: list[Technology] = Field(default_factory=list)
    storage_systems: list[StorageSystem] = Field(default_factory=list)
    participants: list[ProjectParticipant] = Field(default_factory=list)
    locations: list[ExtractedMunicipalityMention] = Field(default_factory=list)

    evidence: str | None = None


### Relaciones entre activos

In [19]:
# Tipo de relación entre activos energéticos.
class AssetRelationType(str, Enum):
    HYBRIDIZES_WITH = "hybridizes_with"
    ADDS_TECHNOLOGY_TO = "adds_technology_to"
    MODIFIES = "modifies"
    EXPANDS = "expands"
    REPOWERS = "repowers"
    ADDS_STORAGE_TO = "adds_storage_to"
    SHARES_GRID_ACCESS_WITH = "shares_grid_access_with"
    ASSOCIATED_WITH = "associated_with"
    SAME_PROJECT_GROUP_AS = "same_project_group_as"
    UNKNOWN = "unknown"

In [20]:
# Relación explícita entre activos.
class AssetRelation(BaseModel):
    source_asset_id: str
    target_asset_id: str
    relation_type: AssetRelationType
    evidence: str | None = None

### Evolución del proyecto

In [21]:
# Tipo de evolución material del proyecto.
class LifecycleEventType(str, Enum):
    NEW_PROJECT = "new_project"
    HYBRIDIZATION = "hybridization"
    MODIFICATION = "modification"
    EXPANSION = "expansion"
    REPOWERING = "repowering"
    STORAGE_ADDITION = "storage_addition"
    EVACUATION_INFRASTRUCTURE = "evacuation_infrastructure"
    OWNERSHIP_CHANGE = "ownership_change"
    OTHER = "other"
    UNKNOWN = "unknown"

In [22]:
# Evento principal de ciclo de vida del proyecto.
# Debería haber un LifecycleEvent por cada objeto administrativo (activo).
class ProjectLifecycleEvent(BaseModel):
    event_type: LifecycleEventType = LifecycleEventType.UNKNOWN
    administrative_actions: list[AdministrativeAction] = Field(default_factory=list)
    assets: list[EnergyAsset] = Field(default_factory=list)
    asset_relations: list[AssetRelation] = Field(default_factory=list)
    event_summary: str | None = None
    evidence: str | None = None

### Documento BOE extraído

In [23]:
class BOEProjectExtraction(BaseModel):
    identificador_boe: str
    fecha_publicacion: date | None = None

    relevancia_energetica: RelevanciaEnergetica
    es_relevante_para_proyecto: bool
    relevance_reason: str | None = None

    lifecycle_events: list[ProjectLifecycleEvent] = Field(default_factory=list)

    extraction_notes: str | None = None

### Notas para la estimación objetiva de la confianza

In [24]:
# TODO: Estimar objetivamente la confianza de la extracción con IA
# confidence = 1.0

# confidence = 1.0

# if project_name is None:
#     confidence -= 0.2

# if promoter is None:
#     confidence -= 0.1

# if len(main_events) == 0:
#     confidence -= 0.3

# if len(locations) == 0:
#     confidence -= 0.1

# if project_name is None:
#     confidence -= 0.2

# if promoter is None:
#     confidence -= 0.1

# if len(main_events) == 0:
#     confidence -= 0.3

# if len(locations) == 0:
#     confidence -= 0.1

# df[
#     (df["relevance_confidence"] < 0.7)
#     | (df["extraction_confidence"] < 0.7)
# ]

## 3. Agente

### Build Agent

In [25]:
def build_agent(
    model,
    output_type: type[BaseModel],
    instructions: str,
    *,
    retries: int = 3,
) -> Agent:
    return Agent(
        model,
        output_type=output_type,
        instructions=instructions,
        retries=retries,
    )

In [26]:
# Para Ollama

from pydantic_ai.models.ollama import OllamaModel
from pydantic_ai.providers.ollama import OllamaProvider


def build_ollama_model(model_name: str) -> OllamaModel:
    return OllamaModel(
        model_name,
        provider=OllamaProvider(
            base_url="http://localhost:11434/v1",
        ),
    )

### Modelos disponibles

In [27]:
# MODEL_PROVIDER = "ollama"
MODEL_PROVIDER = "gemini"

In [28]:
if MODEL_PROVIDER == "gemini":
    AI_MODEL_NAME = "google:gemini-2.5-flash"
    AI_MODEL = AI_MODEL_NAME

elif MODEL_PROVIDER == "ollama":
    AI_MODEL_NAME = "qwen3:8b"
    AI_MODEL = build_ollama_model(AI_MODEL_NAME)

else:
    raise ValueError(
        f"Proveedor de modelo no soportado: {MODEL_PROVIDER}"
    )

In [29]:
agent = Agent(
    AI_MODEL,
    output_type=BOEProjectExtraction,
    instructions=INSTRUCTIONS,
    retries=3
)

## 4. Extracción con IA

In [30]:
TEXT_LIMIT = 4000

### Funciones

#### Construcción de prompt

In [31]:
def build_prompt(
    row: pd.Series,
    text_limit: int = TEXT_LIMIT,
) -> str:
    return f"""
    Identificador BOE: {row["identificador"]}
    Fecha publicación: {row["fecha_publicacion"]}
    Título: {row["titulo"]}

    Texto:
    {row["texto_limpio"][:text_limit]}
"""

#### Registro de extracción

In [32]:
def build_ai_extraction_record(
    row: pd.Series,
    extraction: BOEProjectExtraction,
    model_name: str,
) -> dict:
    """
    Construye un registro tabular a partir de una extracción IA validada.

    La función transforma el objeto Pydantic `BOEProjectExtraction` en una
    fila apta para almacenarse en la capa silver `boe_ai_extractions`.

    Parameters
    ----------
    row : pd.Series
        Fila original del documento BOE procedente de `boe_candidates_docs_text`.
        Debe contener, al menos, la columna `titulo`.
    extraction : BOEProjectExtraction
        Resultado estructurado devuelto por el agente y validado por Pydantic.
    model_name : str
        Nombre del modelo utilizado para generar la extracción.

    Returns
    -------
    dict
        Registro con metadatos de extracción y JSON completo serializado.
    """
    return {
        "identificador_boe": extraction.identificador_boe,
        # TODO: Borrar cuando ejecute todo de cero
        "fecha_publicacion": pd.to_datetime(extraction.fecha_publicacion),
        # "fecha_publicacion": extraction.fecha_publicacion
        ##
        "titulo": row["titulo"],
        "extraction_json": extraction.model_dump_json(),
        "extracted_at": datetime.now(timezone.utc).isoformat(),
        "model_name": model_name,
        "extraction_status": "ok",
        "parse_error": None,
    }

In [33]:
def build_ai_error_record(
    row: pd.Series,
    model_name: str,
    error: Exception,
) -> dict:
    """
    Construye un registro de error cuando falla la extracción IA.
    """
    return {
        "identificador_boe": row["identificador"],
        "fecha_publicacion": row["fecha_publicacion"],
        "titulo": row["titulo"],
        "extraction_json": None,
        "extracted_at": datetime.now(timezone.utc).isoformat(),
        "model_name": model_name,
        "extraction_status": "error",
        "parse_error": str(error),
    }

#### Upsert incremental

In [34]:
def upsert_ai_extractions(
    new_ai_extractions: pd.DataFrame,
    output_path: Path,
) -> pd.DataFrame:
    """
    Inserta o actualiza extracciones IA en un Parquet acumulado.

    Si el fichero ya existe, concatena las nuevas extracciones con las
    existentes y conserva la versión más reciente de cada `identificador_boe`.
    Si no existe, crea el fichero desde cero.

    Esta función debe usarse solo para la tabla fuente
    `boe_ai_extractions.parquet`. Las tablas derivadas deben regenerarse
    a partir de esta tabla, no actualizarse incrementalmente.

    Parameters
    ----------
    new_ai_extractions : pd.DataFrame
        Nuevas extracciones IA a incorporar. Debe contener la columna
        `identificador_boe`.
    output_path : Path
        Ruta del Parquet acumulado de salida.

    Returns
    -------
    pd.DataFrame
        DataFrame completo actualizado tras aplicar el upsert.
    """
    if output_path.exists():
        existing_ai_extractions = pd.read_parquet(output_path)

        ai_extractions = pd.concat(
            [existing_ai_extractions, new_ai_extractions],
            ignore_index=True,
        )

        ai_extractions = ai_extractions.drop_duplicates(
            subset=["identificador_boe"],
            keep="last",
        )
    else:
        ai_extractions = new_ai_extractions.copy()

    # TODO: Borrar cuando ejecute todo de cero
    ai_extractions["fecha_publicacion"] = pd.to_datetime(
        ai_extractions["fecha_publicacion"],
        errors="coerce",
        )
    ##

    save_parquet(ai_extractions, output_path)

    return ai_extractions

### Cargar candidatos BOE

In [35]:
df = pd.read_parquet(BOE_CANDIDATES_DOCS_TEXT_PATH)

df = df.loc[df["xml_status"] == "ok"].copy()

### Filtrar BOEs ya han sido procesados

In [36]:
if BOE_AI_EXTRACTIONS_PATH.exists():
    ai_extractions = pd.read_parquet(
        BOE_AI_EXTRACTIONS_PATH
    )

    processed_ids = set(
        ai_extractions.loc[
            ai_extractions["extraction_status"] == "ok",
            "identificador_boe",
        ]
    )
else:
    ai_extractions = pd.DataFrame()
    processed_ids = set()

print(f"{len(processed_ids)=}")

len(processed_ids)=0


In [37]:
pending_df = df.loc[
    ~df["identificador"].isin(processed_ids)
].copy()

print(
    f"{len(processed_ids)=}"
)
print(
    f"{len(pending_df)=}"
)

len(processed_ids)=0
len(pending_df)=1266


### Seleccionar proyectos de test

#### PE Badulaque

In [66]:
target_ids_badulaque = [
    "BOE-B-2021-32560",
    "BOE-A-2023-2598",
    "BOE-A-2023-10306",
    "BOE-B-2023-19082",
    "BOE-A-2024-16664",
]

#### FV Andévalo

Buenos casos de prueba para comprobar que el sistema no agrupa proyectos distintos simplemente porque comparten infraestructura de evacuación o mencionan FV Andévalo.

In [67]:
target_ids_andevalo = [
    # Proyecto FV Andévalo e hibridaciones
    "BOE-B-2024-26379",
    "BOE-A-2025-18285",
    "BOE-B-2026-3596",

    # Antecedentes y referencias indirectas
    "BOE-A-2022-24404",  # FV Majal Alto
    "BOE-A-2024-9608",   # FV La Puebla 1
    "BOE-A-2025-26110",  # FV La Puebla 1
    "BOE-A-2026-7629",   # FV La Puebla 3 y 4
    "BOE-A-2026-13454",  # FV La Puebla 3
]

Go on...

In [ ]:
target_ids = target_ids_badulaque + target_ids_andevalo

['BOE-B-2021-32560',
 'BOE-A-2023-2598',
 'BOE-A-2023-10306',
 'BOE-B-2023-19082',
 'BOE-A-2024-16664',
 'BOE-B-2024-26379',
 'BOE-A-2025-18285',
 'BOE-B-2026-3596',
 'BOE-A-2022-24404',
 'BOE-A-2024-9608',
 'BOE-A-2025-26110',
 'BOE-A-2026-7629',
 'BOE-A-2026-13454']

In [ ]:
df_test_initial = df.loc[
    df["identificador"].isin(target_ids)
].copy()

df_test_pending = df_test_initial.loc[
    ~df_test_initial["identificador"].isin(processed_ids)
].copy()

print(f"{len(df_test_initial)=}")
print(f"{len(df_test_pending)=}")

len(df_test_initial)=5
len(df_test_pending)=5


## 5. Extracción con agente sobre los pendientes

In [ ]:
records = []
last_extraction = None

In [53]:
for _, row in df_test_pending.iterrows():
    prompt = build_prompt(row)

    try:
        result = await agent.run(prompt)
        extraction = result.output

        record = build_ai_extraction_record(
            row=row,
            extraction=extraction,
            model_name=AI_MODEL_NAME,
        )

    except Exception as exc:
        record = build_ai_error_record(
            row=row,
            model_name=AI_MODEL_NAME,
            error=exc,
        )

    records.append(record)

In [54]:
new_ai_extractions = pd.DataFrame(records)

display(new_ai_extractions)

,identificador_boe,fecha_publicacion,titulo,extraction_json,extracted_at,model_name,extraction_status,parse_error
0,BOE-B-2021-32560,2021-07-07,Anuncio del Área de Industria y Energía de la ...,"{""identificador_boe"":""BOE-B-2021-32560"",""fecha...",2026-06-24T17:42:00.923451+00:00,google:gemini-2.5-flash,ok,None
1,BOE-A-2023-2598,2023-01-31,"Resolución de 23 de enero de 2023, de la Direc...","{""identificador_boe"":""BOE-A-2023-2598"",""fecha_...",2026-06-24T17:42:11.659556+00:00,google:gemini-2.5-flash,ok,None
2,BOE-A-2023-10306,2023-04-28,"Resolución de 17 de abril de 2023, de la Direc...","{""identificador_boe"":""BOE-A-2023-10306"",""fecha...",2026-06-24T17:42:20.837569+00:00,google:gemini-2.5-flash,ok,None
3,BOE-B-2023-19082,2023-06-22,Anuncio del Área Funcional de Industria y Ener...,"{""identificador_boe"":""BOE-B-2023-19082"",""fecha...",2026-06-24T17:42:34.770336+00:00,google:gemini-2.5-flash,ok,None
4,BOE-A-2024-16664,2024-08-10,"Resolución de 22 de julio de 2024, de la Direc...","{""identificador_boe"":""BOE-A-2024-16664"",""fecha...",2026-06-24T17:42:49.391456+00:00,google:gemini-2.5-flash,ok,None


In [55]:
ai_extractions = upsert_ai_extractions(
    new_ai_extractions,
    BOE_AI_EXTRACTIONS_PATH,
)

display(ai_extractions)

,identificador_boe,fecha_publicacion,titulo,extraction_json,extracted_at,model_name,extraction_status,parse_error
0,BOE-A-2022-24404,2022-12-30,"Resolución de 22 de diciembre de 2022, de la D...","{""identificador_boe"":""BOE-A-2022-24404"",""fecha...",2026-06-24T17:38:11.325182+00:00,google:gemini-2.5-flash,ok,None
1,BOE-A-2024-9608,2024-05-13,"Resolución de 6 de mayo de 2024, de la Direcci...","{""identificador_boe"":""BOE-A-2024-9608"",""fecha_...",2026-06-24T17:38:20.818257+00:00,google:gemini-2.5-flash,ok,None
2,BOE-B-2024-26379,2024-07-13,Anuncio del Área de Industria y Energía de la ...,"{""identificador_boe"":""BOE-B-2024-26379"",""fecha...",2026-06-24T17:38:33.504617+00:00,google:gemini-2.5-flash,ok,None
3,BOE-A-2025-18285,2025-09-15,"Resolución de 7 de agosto de 2025, de la Direc...","{""identificador_boe"":""BOE-A-2025-18285"",""fecha...",2026-06-24T17:38:43.770102+00:00,google:gemini-2.5-flash,ok,None
4,BOE-A-2025-26110,2025-12-19,"Resolución de 17 de noviembre de 2025, de la D...","{""identificador_boe"":""BOE-A-2025-26110"",""fecha...",2026-06-24T17:39:05.694965+00:00,google:gemini-2.5-flash,ok,None
5,BOE-B-2026-3596,2026-02-07,Anuncio del Área de Industria y Energía de la ...,"{""identificador_boe"":""BOE-B-2026-3596"",""fecha_...",2026-06-24T17:39:16.502758+00:00,google:gemini-2.5-flash,ok,None
6,BOE-A-2026-7629,2026-04-03,"Resolución de 17 de marzo de 2026, de la Direc...","{""identificador_boe"":""BOE-A-2026-7629"",""fecha_...",2026-06-24T17:39:36.239255+00:00,google:gemini-2.5-flash,ok,None
7,BOE-A-2026-13454,2026-06-20,"Resolución de 3 de junio de 2026, de la Direcc...","{""identificador_boe"":""BOE-A-2026-13454"",""fecha...",2026-06-24T17:39:59.286939+00:00,google:gemini-2.5-flash,ok,None
8,BOE-B-2021-32560,2021-07-07,Anuncio del Área de Industria y Energía de la ...,"{""identificador_boe"":""BOE-B-2021-32560"",""fecha...",2026-06-24T17:42:00.923451+00:00,google:gemini-2.5-flash,ok,None
9,BOE-A-2023-2598,2023-01-31,"Resolución de 23 de enero de 2023, de la Direc...","{""identificador_boe"":""BOE-A-2023-2598"",""fecha_...",2026-06-24T17:42:11.659556+00:00,google:gemini-2.5-flash,ok,None


## 6. Chequeo de todo lo extraído

In [56]:
extraido = pd.read_parquet(BOE_AI_EXTRACTIONS_PATH)

import json

json_data = extraido.loc[
    extraido["identificador_boe"] == "BOE-B-2021-32560",
    "extraction_json",
].iloc[0]

print(json.dumps(json.loads(json_data), indent=2, ensure_ascii=False))

{
  "identificador_boe": "BOE-B-2021-32560",
  "fecha_publicacion": "2021-07-07",
  "relevancia_energetica": "relevante",
  "es_relevante_para_proyecto": true,
  "relevance_reason": "El documento somete a información pública un Estudio de Impacto Ambiental y la solicitud de Autorización Administrativa Previa para el Parque Eólico Badulaque de 90 MW.",
  "lifecycle_events": [
    {
      "event_type": "new_project",
      "administrative_actions": [
        {
          "procedure_stage": "informacion_publica",
          "procedure_decision": "sometido_informacion_publica",
          "evidence": "se somete al trámite de información pública, de forma conjunta, el Estudio de Impacto Ambiental y la solicitud de Autorización Administrativa Previa"
        }
      ],
      "assets": [
        {
          "local_asset_id": "asset_1",
          "name": "Parque eólico Badulaque",
          "aliases": [],
          "role_in_event": "main_asset",
          "status_in_document": "under_permitting",

## 7. Flattening data in field extraction_json

### Funciones

#### flatten_lifecycle_events

In [57]:
def flatten_lifecycle_events(ai_extractions: pd.DataFrame) -> pd.DataFrame:
    records = []

    for _, row in ai_extractions.iterrows():
        extraction = BOEProjectExtraction.model_validate_json(
            row["extraction_json"]
        )

        for event_idx, event in enumerate(
            extraction.lifecycle_events,
            start=1,
        ):
            event_id = f"{extraction.identificador_boe}_event_{event_idx}"

            records.append(
                {
                    "event_id": event_id,
                    "identificador_boe": extraction.identificador_boe,
                    "fecha_publicacion": extraction.fecha_publicacion,
                    "event_index": event_idx,
                    "event_type": event.event_type.value,
                    "event_summary": event.event_summary,
                    "evidence": event.evidence,
                }
            )

    return pd.DataFrame(records)

#### flatten_administrative_actions

In [58]:
def flatten_administrative_actions(ai_extractions: pd.DataFrame) -> pd.DataFrame:
    records = []

    for _, row in ai_extractions.iterrows():
        extraction = BOEProjectExtraction.model_validate_json(
            row["extraction_json"]
        )

        for event_idx, event in enumerate(
            extraction.lifecycle_events,
            start=1,
        ):
            event_id = f"{extraction.identificador_boe}_event_{event_idx}"

            for action_idx, action in enumerate(
                event.administrative_actions,
                start=1,
            ):
                action_id = (
                    f"{extraction.identificador_boe}"
                    f"_event_{event_idx}"
                    f"_action_{action_idx}"
                )

                records.append(
                    {
                        "action_id": action_id,
                        "event_id": event_id,
                        "identificador_boe": extraction.identificador_boe,
                        "fecha_publicacion": extraction.fecha_publicacion,
                        "action_index": action_idx,
                        "procedure_stage": action.procedure_stage.value,
                        "procedure_decision": action.procedure_decision.value,
                        "evidence": action.evidence,
                    }
                )

    return pd.DataFrame(records)

#### flatten_asset_mentions

In [59]:
def flatten_asset_mentions(ai_extractions: pd.DataFrame) -> pd.DataFrame:
    records = []

    for _, row in ai_extractions.iterrows():
        extraction = BOEProjectExtraction.model_validate_json(row["extraction_json"])

        for event_idx, event in enumerate(extraction.lifecycle_events, start=1):
            event_id = f"{extraction.identificador_boe}_event_{event_idx}"

            for asset in event.assets:
                asset_mention_id = f"{event_id}_{asset.local_asset_id}"

                records.append(
                    {
                        "asset_mention_id": asset_mention_id,
                        "event_id": event_id,
                        "identificador_boe": extraction.identificador_boe,
                        "fecha_publicacion": extraction.fecha_publicacion,
                        "local_asset_id": asset.local_asset_id,
                        "asset_name": asset.name,
                        "asset_name_norm": normalize_text(asset.name) if asset.name else None,
                        "role_in_event": asset.role_in_event.value,
                        "status_in_document": asset.status_in_document.value,
                        "evidence": asset.evidence,
                    }
                )

    return pd.DataFrame(records)

#### flatten_asset_technologies

In [60]:
def flatten_asset_technologies(ai_extractions: pd.DataFrame) -> pd.DataFrame:
    records = []

    for _, row in ai_extractions.iterrows():
        extraction = BOEProjectExtraction.model_validate_json(row["extraction_json"])

        for event_idx, event in enumerate(extraction.lifecycle_events, start=1):
            event_id = f"{extraction.identificador_boe}_event_{event_idx}"

            for asset in event.assets:
                asset_mention_id = f"{event_id}_{asset.local_asset_id}"

                for tech_idx, tech in enumerate(asset.technologies, start=1):
                    records.append(
                        {
                            "asset_technology_id": f"{asset_mention_id}_tech_{tech_idx}",
                            "asset_mention_id": asset_mention_id,
                            "event_id": event_id,
                            "identificador_boe": extraction.identificador_boe,
                            "technology_type": tech.technology_type.value,
                            "installed_power_mw": tech.installed_power_mw,
                            "peak_power_mwp": tech.peak_power_mwp,
                            "description": tech.description,
                            "power_normalization_note": tech.power_normalization_note,
                        }
                    )

    return pd.DataFrame(records)

#### flatten_asset_participants

In [61]:
def flatten_asset_participants(ai_extractions: pd.DataFrame) -> pd.DataFrame:
    records = []

    for _, row in ai_extractions.iterrows():
        extraction = BOEProjectExtraction.model_validate_json(row["extraction_json"])

        for event_idx, event in enumerate(extraction.lifecycle_events, start=1):
            event_id = f"{extraction.identificador_boe}_event_{event_idx}"

            for asset in event.assets:
                asset_mention_id = f"{event_id}_{asset.local_asset_id}"

                for participant_idx, participant in enumerate(asset.participants, start=1):
                    records.append(
                        {
                            "asset_participant_id": f"{asset_mention_id}_participant_{participant_idx}",
                            "asset_mention_id": asset_mention_id,
                            "event_id": event_id,
                            "identificador_boe": extraction.identificador_boe,
                            "participant_name": participant.name,
                            "participant_name_norm": normalize_text(participant.name) if participant.name else None,
                            "participant_role": participant.role.value,
                            "evidence": participant.evidence,
                        }
                    )

    return pd.DataFrame(records)

#### flatten_asset_locations

In [ ]:
# TODO: La IA devolverá variantes textuales distintas.

def flatten_asset_locations(ai_extractions: pd.DataFrame) -> pd.DataFrame:
    records = []

    for _, row in ai_extractions.iterrows():
        extraction = BOEProjectExtraction.model_validate_json(row["extraction_json"])

        for event_idx, event in enumerate(extraction.lifecycle_events, start=1):
            event_id = f"{extraction.identificador_boe}_event_{event_idx}"

            for asset in event.assets:
                asset_mention_id = f"{event_id}_{asset.local_asset_id}"

                for location_idx, loc in enumerate(asset.locations, start=1):
                    records.append({
                        "asset_location_id": f"{asset_mention_id}_location_{location_idx}",
                        "asset_mention_id": asset_mention_id,
                        "event_id": event_id,
                        "identificador_boe": extraction.identificador_boe,
                        "municipality_raw": loc.municipality_name,
                        "municipality_raw_norm": normalize_text(loc.municipality_name) if loc.municipality_name else None,
                        "province_hint_raw": loc.province_hint,
                        "province_hint_raw_norm": normalize_text(loc.province_hint) if loc.province_hint else None,
                        "autonomous_community_hint_raw": loc.autonomous_community_hint,
                        "autonomous_community_hint_raw_norm": normalize_text(loc.autonomous_community_hint) if loc.autonomous_community_hint else None,
                        "location_evidence": loc.evidence,
                    })

    return pd.DataFrame(records)

#### flatten_asset_aliases

In [63]:
def flatten_asset_aliases(ai_extractions: pd.DataFrame) -> pd.DataFrame:
    records = []

    for _, row in ai_extractions.iterrows():
        extraction = BOEProjectExtraction.model_validate_json(row["extraction_json"])

        for event_idx, event in enumerate(extraction.lifecycle_events, start=1):
            event_id = f"{extraction.identificador_boe}_event_{event_idx}"

            for asset in event.assets:
                asset_mention_id = f"{event_id}_{asset.local_asset_id}"

                for alias_idx, alias in enumerate(asset.aliases, start=1):
                    records.append(
                        {
                            "asset_alias_id": f"{asset_mention_id}_alias_{alias_idx}",
                            "asset_mention_id": asset_mention_id,
                            "event_id": event_id,
                            "identificador_boe": extraction.identificador_boe,
                            "alias": alias,
                            "alias_norm": normalize_text(alias) if alias else None,
                        }
                    )

    return pd.DataFrame(records)

#### flatten_asset_relation_mentions

In [64]:
def flatten_asset_relation_mentions(ai_extractions: pd.DataFrame) -> pd.DataFrame:
    records = []

    for _, row in ai_extractions.iterrows():
        extraction = BOEProjectExtraction.model_validate_json(row["extraction_json"])

        for event_idx, event in enumerate(extraction.lifecycle_events, start=1):
            event_id = f"{extraction.identificador_boe}_event_{event_idx}"

            for relation_idx, relation in enumerate(event.asset_relations, start=1):
                records.append(
                    {
                        "relation_mention_id": (
                            f"{extraction.identificador_boe}"
                            f"_event_{event_idx}"
                            f"_relation_{relation_idx}"
                        ),
                        "event_id": event_id,
                        "identificador_boe": extraction.identificador_boe,
                        "fecha_publicacion": extraction.fecha_publicacion,
                        "source_asset_mention_id": (
                            f"{extraction.identificador_boe}"
                            f"_event_{event_idx}_{relation.source_asset_id}"
                        ),
                        "target_asset_mention_id": (
                            f"{extraction.identificador_boe}"
                            f"_event_{event_idx}_{relation.target_asset_id}"
                        ),
                        "source_local_asset_id": relation.source_asset_id,
                        "target_local_asset_id": relation.target_asset_id,
                        "relation_type": relation.relation_type.value,
                        "evidence": relation.evidence,
                    }
                )

    return pd.DataFrame(records)

### Prueba

In [73]:
lifecycle_events = flatten_lifecycle_events(ai_extractions)
administrative_actions = flatten_administrative_actions(ai_extractions)
asset_mentions = flatten_asset_mentions(ai_extractions)
asset_technologies = flatten_asset_technologies(ai_extractions)
asset_participants = flatten_asset_participants(ai_extractions)
asset_locations = flatten_asset_locations(ai_extractions)
asset_aliases = flatten_asset_aliases(ai_extractions)
asset_relation_mentions = flatten_asset_relation_mentions(ai_extractions)

In [74]:
save_parquet(lifecycle_events, LIFECYCLE_EVENTS_PATH)
save_parquet(administrative_actions, ADMINISTRATIVE_ACTIONS_PATH)
save_parquet(asset_mentions, ASSET_MENTIONS_PATH)
save_parquet(asset_technologies, ASSET_TECHNOLOGIES_PATH)
save_parquet(asset_participants, ASSET_PARTICIPANTS_PATH)
save_parquet(asset_locations, ASSET_LOCATIONS_PATH)
save_parquet(asset_aliases, ASSET_ALIASES_PATH)
save_parquet(asset_relation_mentions, ASSET_RELATION_MENTIONS_PATH)

In [75]:
lifecycle_events = pd.read_parquet(LIFECYCLE_EVENTS_PATH)
administrative_actions = pd.read_parquet(ADMINISTRATIVE_ACTIONS_PATH)
asset_mentions = pd.read_parquet(ASSET_MENTIONS_PATH)
asset_technologies = pd.read_parquet(ASSET_TECHNOLOGIES_PATH)
asset_participants = pd.read_parquet(ASSET_PARTICIPANTS_PATH)
asset_locations = pd.read_parquet(ASSET_LOCATIONS_PATH)
asset_aliases = pd.read_parquet(ASSET_ALIASES_PATH)
asset_relation_mentions = pd.read_parquet(ASSET_RELATION_MENTIONS_PATH)

In [ ]:
display(lifecycle_events)
display(administrative_actions)
display(asset_mentions)
display(asset_technologies)
display(asset_participants)
display(asset_locations)
display(asset_aliases)
display(asset_relation_mentions)

In [77]:
asset_locations.loc[
    asset_locations["municipality_raw_norm"].str.contains("pontes", na=False)
]

,asset_location_id,asset_mention_id,event_id,identificador_boe,municipality_raw,municipality_raw_norm,province_hint_raw,province_hint_raw_norm,autonomous_community_hint_raw,autonomous_community_hint_raw_norm,location_evidence
35,BOE-B-2021-32560_event_1_asset_1_location_1,BOE-B-2021-32560_event_1_asset_1,BOE-B-2021-32560_event_1,BOE-B-2021-32560,As Pontes,as pontes,A Coruña,a coruna,None,None,"Municipios afectados: As Pontes, As Somozas, C..."
51,BOE-A-2023-2598_event_1_asset_1_location_6,BOE-A-2023-2598_event_1_asset_1,BOE-A-2023-2598_event_1,BOE-A-2023-2598,As Pontes de García Rodríguez,as pontes de garcia rodriguez,A Coruña,a coruna,None,None,As Pontes de García Rodríguez
57,BOE-A-2023-10306_event_1_asset_1_location_6,BOE-A-2023-10306_event_1_asset_1,BOE-A-2023-10306_event_1,BOE-A-2023-10306,As Pontés,as pontes,A Coruña,a coruna,Galicia,galicia,y As Pontés (A Coruña)
63,BOE-B-2023-19082_event_1_asset_1_location_6,BOE-B-2023-19082_event_1_asset_1,BOE-B-2023-19082_event_1,BOE-B-2023-19082,As Pontes de García Rodríguez,as pontes de garcia rodriguez,A Coruña,a coruna,Galicia,galicia,"términos municipales de Valdoviño, Cedeira, Ce..."
69,BOE-A-2024-16664_event_1_asset_1_location_6,BOE-A-2024-16664_event_1_asset_1,BOE-A-2024-16664_event_1,BOE-A-2024-16664,As Pontes de García Rodríguez,as pontes de garcia rodriguez,A Coruña,a coruna,None,None,ubicados en los términos municipales de Valdov...
